In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import random
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler

from typing import List, Optional, Sequence, Tuple, Dict
from itertools import combinations


from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import timm
from tqdm import tqdm

# Preprocessing

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()


SEED = 43

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

data_dir            = "/path/to/ISIC_DATA"
train_dir            = "/path/to/ISIC_DATA/ISIC_2019_Training_Input"
test_dir             = "/path/to/ISIC_DATA/ISIC_2019_Test_Input"
train_truth_file     = "ISIC_2019_Training_GroundTruth.csv"
test_truth_file      = "ISIC_2019_Test_GroundTruth.csv"
train_metadata_file  = "ISIC_2019_Training_Metadata.csv"

# -------------------------------------------------------------- classes
CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES = len(CLASS_NAMES)
UNK_LABEL = NUM_CLASSES

# ----------------------------------------------------------------- data
IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4

# ---------------------------------------------------------------- model
ARCH = "tf_efficientnetv2_m.in21k_ft_in1k"
DROPOUT = 0.4
DROP_PATH_RATE = 0.2
CODEBOOK_K_VALUES = (1, 7)

# ------------------------------------------------------------- training
NUM_EPOCHS = 35
LR = 1e-4
WEIGHT_DECAY = 0.01



In [ ]:
train_gt = pd.read_csv(os.path.join(data_dir, train_truth_file))
test_gt  = pd.read_csv(os.path.join(data_dir, test_truth_file))

# Create single_label for training (exclude image & UNK)
train_labels = train_gt.iloc[:, 1:9]
train_gt['single_label'] = train_labels.values.argmax(axis=1)

# For test (include UNK)
test_labels = test_gt.iloc[:, 1:10]
test_gt['single_label'] = test_labels.values.argmax(axis=1)


meta = pd.read_csv(os.path.join(data_dir, train_metadata_file))

# Merge lesion_id from metadata into train_gt (left join, keyed on image name)
train_gt = train_gt.merge(meta[['image', 'lesion_id']], on='image', how='left')
train_gt['group_id'] = train_gt['lesion_id'].fillna(train_gt['image'])

# Stratified GROUP Train/Val Split

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(
    sgkf.split(train_gt, train_gt['single_label'], groups=train_gt['group_id'])
)

train_df = train_gt.iloc[train_idx].reset_index(drop=True).drop(columns=['group_id', 'lesion_id'])
val_df   = train_gt.iloc[val_idx].reset_index(drop=True).drop(columns=['group_id', 'lesion_id'])

train_groups = set(train_gt.loc[train_idx, 'group_id'])
val_groups   = set(train_gt.loc[val_idx, 'group_id'])
overlap = train_groups & val_groups
print(f"Overlapping groups between train/val: {len(overlap)}")

In [ ]:
class CenterSquareCrop:
    def __call__(self, img):
        w, h = img.size
        m = min(w, h)
        left, top = (w-m)//2, (h-m)//2
        return img.crop((left, top, left+m, top+m))


IMAGENET_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STD = [0.5, 0.5, 0.5]

IMAGENET_MEAN_FILL = tuple(
    round(x * 255) for x in IMAGENET_MEAN
)

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    CenterSquareCrop(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees = 30, translate=(0.01,0.01), scale=(1.0,1.05), fill=IMAGENET_MEAN_FILL),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    CenterSquareCrop(),
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])


class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        name = self.df.iloc[idx]['image'] + '.jpg'
        img = Image.open(os.path.join(self.img_dir, name))
        label = int(self.df.iloc[idx]['single_label'])
        if self.transform:
            img = self.transform(img)
        return img, label


train_ds = ISICDataset(train_df, train_dir, train_transform)
val_ds   = ISICDataset(val_df,   train_dir, val_transform)
test_ds  = ISICDataset(test_gt,  test_dir,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, worker_init_fn=seed_worker, generator=g,)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,num_workers=4, pin_memory=True)

## Training Routine

### Codebook

In [ ]:
Spec = Tuple[str, List[str], List[str]]  # (head_name, positive_classes, negative_classes)

def make_one_vs_k_specs(class_names: Sequence[str], k: int) -> List[Spec]:
   
    C = len(class_names)
    if not 1 <= k <= C - 1:
        raise ValueError(f"k must be in [1, {C - 1}], got {k}")

    class_order = {c: i for i, c in enumerate(class_names)}
    specs: List[Spec] = []

    for pos in class_names:
        remaining = [c for c in class_names if c != pos]
        for neg_subset in combinations(remaining, k):
            neg_sorted = sorted(neg_subset, key=class_order.__getitem__)
            name = f"1v{k}_{pos}_vs_{'-'.join(neg_sorted)}"
            specs.append((name, [pos], list(neg_sorted)))

    return specs


def build_codebook(
    class_names: Sequence[str],
    k_values: Sequence[int] = (7, 6, 5),
) -> Tuple[torch.Tensor, List[str]]:

    C = len(class_names)
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    specs: List[Spec] = []
    for k in k_values:
        specs += make_one_vs_k_specs(class_names, k)

    columns: List[torch.Tensor] = []
    head_names: List[str] = []
    seen: set = set()

    for head_name, pos_classes, neg_classes in specs:
        col = torch.zeros(C, dtype=torch.float32)
        for c in pos_classes:
            col[class_to_idx[c]] = 1.0
        for c in neg_classes:
            col[class_to_idx[c]] = -1.0

        key = tuple(col.tolist())
        flipped_key = tuple((-col).tolist())
        if key in seen or flipped_key in seen:
            continue

        seen.add(key)
        columns.append(col)
        head_names.append(head_name)

    codebook = torch.stack(columns, dim=1)  # [C, H]
    return codebook, head_names


### Model

In [ ]:
class EfficientNetECOCModel(nn.Module):
    def __init__(
        self,
        codebook: torch.Tensor,           # [C, L] ternary +1/-1/0
        column_names: Optional[Sequence[str]] = None,
        arch: str = 'tf_efficientnetv2_m.in21k_ft_in1k',
        pretrained: bool = True,
        dropout: float = 0.2,
        drop_path_rate: float = 0.0,
        head_hidden_dim: Optional[int] = None,  # None = single Linear per head

    ):
        super().__init__()

        codebook = torch.as_tensor(codebook, dtype=torch.float32)
        if codebook.ndim != 2:
            raise ValueError(f"codebook must be 2D [num_classes, code_length], got {tuple(codebook.shape)}")
        unique_vals = set(codebook.unique().tolist())
        if not unique_vals.issubset({-1.0, 0.0, 1.0}):
            raise ValueError(f"codebook must contain only -1, 0, +1. Found: {sorted(unique_vals)}")

        self.num_classes = int(codebook.shape[0])
        self.code_length = int(codebook.shape[1])


        if column_names is None:
            column_names = [f"ECOC_{i:03d}" for i in range(self.code_length)]
        if len(column_names) != self.code_length:
            raise ValueError("column_names length must match code_length.")
        self.column_names: List[str] = list(column_names)

        self.register_buffer("codebook_pm1", codebook)
        self.register_buffer("class_active_counts", (codebook != 0).float().sum(dim=1).clamp_min(1.0))


        self.backbone = timm.create_model(
            arch,
            pretrained=pretrained,
            features_only=True,
            out_indices=(-1,),
            drop_path_rate=drop_path_rate,
        )
        self.feat_dim = self.backbone.feature_info.info[-1]['num_chs']
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        print(f"Backbone feature dim: {self.feat_dim}")
        self.bit_heads = nn.ModuleList([
            self._make_bit_head(dropout, head_hidden_dim) for _ in range(self.code_length)
        ])

    def _make_bit_head(self, dropout: float, head_hidden_dim: Optional[int]) -> nn.Module:
        if head_hidden_dim is None:
            return nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feat_dim, 2))
        return nn.Sequential(
            nn.Linear(self.feat_dim, head_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 2),
        )


    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)[-1]
        return self.pool(feat).float()

    def forward_bit_logits(self, x: torch.Tensor) -> torch.Tensor:
        features = self.extract_features(x)
        logits = [head(features).unsqueeze(1) for head in self.bit_heads]
        return torch.cat(logits, dim=1)  # [B, L, 2]

    def decode_bit_probs(self, p_pos: torch.Tensor) -> torch.Tensor:
        if p_pos.shape[1] != self.code_length:
            raise ValueError(f"p_pos must have shape [B, {self.code_length}], got {tuple(p_pos.shape)}")
        soft_bits = 2.0 * p_pos - 1.0
        class_scores = soft_bits @ self.codebook_pm1.T
        class_scores = class_scores / self.class_active_counts.view(1, -1)
        return class_scores

    def get_bit_probs(self, x: torch.Tensor) -> torch.Tensor:
        """Returns sigmoid probabilities [B, L] — prob that each bit is +1."""
        logits = self.forward_bit_logits(x)  # [B, L, 2]
        # logit[:, :, 1] is the logit for "positive" class
        return torch.sigmoid(logits[:, :, 1] - logits[:, :, 0])  # [B, L]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        p_pos = self.get_bit_probs(x)
        return self.decode_bit_probs(p_pos)


In [ ]:
class ECOCLoss(nn.Module):
    def __init__(self, codebook_pm1: torch.Tensor, head_reduction: str = "mean"):
        super().__init__()
        codebook_pm1 = torch.as_tensor(codebook_pm1, dtype=torch.float32)
        self.register_buffer("codebook_pm1", codebook_pm1)
        self.num_classes = codebook_pm1.shape[0]
        self.code_length = codebook_pm1.shape[1]
        self.head_reduction = head_reduction

    def encode_targets(self, y: torch.Tensor):
        y = y.long()
        valid = (y >= 0) & (y < self.num_classes)
        y_safe = y.clamp(0, self.num_classes - 1)
        target_pm1 = self.codebook_pm1[y_safe]          # [B, L]
        active_mask = (target_pm1 != 0) & valid.view(-1, 1)  # [B, L]
        target_01 = (target_pm1 > 0).float()             # [B, L]
        return target_01, active_mask

    def forward(self, bit_logits: torch.Tensor, y: torch.Tensor):
        target_01, active_mask = self.encode_targets(y)
        device = bit_logits.device

        logit_pos = bit_logits[:, :, 1] - bit_logits[:, :, 0]  # [B, L]
        target_01 = target_01.to(device)
        active_mask_f = active_mask.float().to(device)

        bce_per = F.binary_cross_entropy_with_logits(
            logit_pos, target_01, reduction='none'
        )  # [B, L]

        masked = bce_per * active_mask_f
        per_head_active = active_mask_f.sum(dim=0).clamp_min(1.0)
        head_losses = masked.sum(dim=0) / per_head_active  # [L]

        active_heads = active_mask_f.sum(dim=0) > 0
        head_losses = head_losses[active_heads]

        if self.head_reduction == "mean":
            total = head_losses.mean()
        else:
            total = head_losses.sum()

        return total, {"loss": total.detach()}

In [ ]:
def compute_ecoc_epoch_metrics(model, p_pos: torch.Tensor, y: torch.Tensor) -> Dict[str, float]:
    valid = (y >= 0) & (y < model.num_classes)
    class_scores = model.decode_bit_probs(p_pos.to(model.codebook_pm1.device))
    preds = class_scores.argmax(dim=1).cpu()

    if valid.any():
        y_true = y[valid].numpy()
        y_pred = preds[valid].numpy()
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        balanced_acc = balanced_accuracy_score(y_true, y_pred)
    else:
        macro_f1 = balanced_acc = 0.0

    return {"class_macro_f1": macro_f1, "class_balanced_accuracy": balanced_acc}

In [ ]:
codebook, head_names = build_codebook(CLASS_NAMES, k_values=CODEBOOK_K_VALUES)

print(f"num_classes={codebook.shape[0]}, num_heads={codebook.shape[1]}")

model = EfficientNetECOCModel(
    codebook=codebook,
    column_names=head_names,
    arch=ARCH,
    pretrained=True,
    dropout=DROPOUT,
    drop_path_rate=DROP_PATH_RATE,
).to(device)

criterion = ECOCLoss(
    codebook_pm1=model.codebook_pm1.detach().cpu(),
    head_reduction='mean',
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler()

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, total_samples = 0.0, 0
    all_p_pos, all_y = [], []

    for inputs, targets in tqdm(loader, desc='Train'):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast():
            bit_logits = model.forward_bit_logits(inputs)  # [B, L, 2]
            loss, _ = criterion(bit_logits.float(), targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * targets.size(0)
        total_samples += targets.size(0)

        with torch.no_grad():
            p_pos = torch.sigmoid(bit_logits.float()[:, :, 1] - bit_logits.float()[:, :, 0])
            all_p_pos.append(p_pos.detach().cpu())
            all_y.append(targets.cpu())

    avg_loss = total_loss / total_samples
    all_p_pos = torch.cat(all_p_pos, dim=0)
    all_y = torch.cat(all_y, dim=0)
    metrics = compute_ecoc_epoch_metrics(model, all_p_pos, all_y)
    return avg_loss, metrics


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_p_pos, all_y = [], []

    for inputs, targets in tqdm(loader, desc='Eval'):
        inputs, targets = inputs.to(device), targets.to(device)
        bit_logits = model.forward_bit_logits(inputs).float()
        loss, _ = criterion(bit_logits, targets)

        total_loss += loss.item() * targets.size(0)
        total_samples += targets.size(0)

        p_pos = torch.sigmoid(bit_logits[:, :, 1] - bit_logits[:, :, 0])
        all_p_pos.append(p_pos.cpu())
        all_y.append(targets.cpu())

    avg_loss = total_loss / total_samples
    all_p_pos = torch.cat(all_p_pos, dim=0)
    all_y = torch.cat(all_y, dim=0)
    metrics = compute_ecoc_epoch_metrics(model, all_p_pos, all_y)
    return avg_loss, metrics

In [ ]:
best_val_f1 = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_metrics = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Macro F1: {train_metrics['class_macro_f1']:.4f} | "
        f"Balanced Acc: {train_metrics['class_balanced_accuracy']:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Macro F1: {val_metrics['class_macro_f1']:.4f} | "
        f"Balanced Acc: {val_metrics['class_balanced_accuracy']:.4f}"
    )
    
    if val_metrics['class_macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['class_macro_f1']
        torch.save(model.state_dict(), 'ecoc_checkpoint.pth')
        print(f"Saved (F1={best_val_f1:.4f})")
        
print("Training complete.")

# Testing:

In [ ]:
model.load_state_dict(torch.load('ecoc_checkpoint.pth', weights_only=True))
model.to(device)

In [ ]:
@torch.no_grad()
def get_predictions_and_uncertainty(
    model,
    bit_logits: torch.Tensor,
):

    bit_logits = bit_logits.float().to(
        model.codebook_pm1.device
    )

    probs = torch.softmax(bit_logits, dim=-1)

    confidence_per_head = probs.max(dim=-1).values
    uncertainty_per_head = 1.0 - confidence_per_head

    p_pos = probs[:, :, 1]

    # Use the model's decoding implementation
    class_scores = model.decode_bit_probs(p_pos)
    preds = class_scores.argmax(dim=1)

    pred_active_mask = (
        model.codebook_pm1[preds] != 0
    ).float()

    pred_denom = (
        pred_active_mask.sum(dim=1)
        .clamp_min(1.0)
    )

    uncertainty = (
        uncertainty_per_head * pred_active_mask
    ).sum(dim=1) / pred_denom

    global_uncertainty = (
        uncertainty_per_head.mean(dim=1)
    )

    soft_bits = 2.0 * p_pos - 1.0
    contrib_weight = (
        soft_bits.abs() * pred_active_mask
    )

    contribution_weighted_uncertainty = (
        uncertainty_per_head * contrib_weight
    ).sum(dim=1) / pred_denom

    return (
        preds,
        uncertainty,
        global_uncertainty,
        contribution_weighted_uncertainty,
    )

In [ ]:
def compute_auroc(labels: np.ndarray, scores: np.ndarray, score_name: str, task_name: str):
    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())

    if n_pos == 0 or n_neg == 0:
        print(f"[{score_name}] Cannot compute {task_name} AUROC: n_positive={n_pos}, n_negative={n_neg}")
        return None

    auroc = roc_auc_score(labels, scores)
    print(f"[{score_name}] {task_name} AUROC: {auroc:.4f}  (n_neg={n_neg}, n_pos={n_pos})")
    return auroc

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    dataloader,
    device,
    num_classes=8,
    unk_label=8,
):
    model.eval()

    all_targets = []
    all_preds = []
    all_contrib_uncertainty = []

    for inputs, targets in tqdm(
        dataloader,
        desc="Evaluating",
        ncols=100,
    ):
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Single forward pass for the complete batch
        bit_logits = model.forward_bit_logits(
            inputs
        ).float()

        predicted, _, _, contrib_uncertainty = (
            get_predictions_and_uncertainty(
                model,
                bit_logits,
            )
        )

        all_targets.extend(
            targets.cpu().numpy()
        )
        all_preds.extend(
            predicted.cpu().numpy()
        )
        all_contrib_uncertainty.extend(
            contrib_uncertainty.cpu().numpy()
        )

    all_targets = np.asarray(all_targets)
    all_preds = np.asarray(all_preds)
    all_contrib_uncertainty = np.asarray(
        all_contrib_uncertainty
    )

    # 1. Classification metrics
    known_mask = all_targets != unk_label

    known_targets = all_targets[known_mask]
    known_preds = all_preds[known_mask]

    macro_f1 = f1_score(
        known_targets,
        known_preds,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc = balanced_accuracy_score(
        known_targets,
        known_preds,
    )

    # 2. UNK-detection AUROC
    is_unk = (
        all_targets == unk_label
    ).astype(int)

    unk_auroc = compute_auroc(
        is_unk,
        all_contrib_uncertainty,
        "contribution-weighted avg",
        "UNK-detection",
    )

    return (
        macro_f1,
        balanced_acc,
        unk_auroc,
    )

In [ ]:
(
    test_macro_f1,
    test_balanced_acc,
    test_unk_auroc,
) = evaluate_model(
    model,
    test_loader,
    device,
)

print(f"Macro F1:                         {test_macro_f1:.4f}")
print(f"Balanced Accuracy:                {test_balanced_acc:.4f}")
print(f"UNK-detection AUROC:              {test_unk_auroc:.4f}")

In [ ]:
@torch.no_grad()
def collect_test_predictions(
    model,
    dataloader,
    device,
):
    model.eval()

    all_preds = []
    all_targets = []
    all_uncertainty = []

    for inputs, targets in tqdm(
        dataloader,
        desc="Collecting test predictions",
        ncols=100,
    ):
        inputs = inputs.to(device)
        targets = targets.to(device)

        bit_logits = model.forward_bit_logits(
            inputs
        ).float()

        preds, _, _, contrib_uncertainty = (
            get_predictions_and_uncertainty(
                model,
                bit_logits,
            )
        )

        all_preds.extend(
            preds.cpu().numpy()
        )
        all_targets.extend(
            targets.cpu().numpy()
        )
        all_uncertainty.extend(
            contrib_uncertainty.cpu().numpy()
        )

    return (
        np.asarray(all_preds),
        np.asarray(all_targets),
        np.asarray(all_uncertainty),
    )

def evaluate_id_after_pruning(
    preds,
    targets,
    uncertainty,
    prune_frac=0.10,
    num_classes=8,
    unk_label=8,
):

    id_mask = targets != unk_label

    preds_id = preds[id_mask]
    targets_id = targets[id_mask]
    uncertainty_id = uncertainty[id_mask]

    n_id = len(targets_id)
    n_prune = int(round(n_id * prune_frac))
    n_keep = n_id - n_prune

    # Ascending uncertainty:
    # least uncertain samples are retained.
    order = np.argsort(uncertainty_id)
    keep_idx = order[:n_keep]

    kept_preds = preds_id[keep_idx]
    kept_targets = targets_id[keep_idx]

    macro_f1_before = f1_score(
        targets_id,
        preds_id,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc_before = (
        balanced_accuracy_score(
            targets_id,
            preds_id,
        )
    )

    macro_f1_after = f1_score(
        kept_targets,
        kept_preds,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc_after = (
        balanced_accuracy_score(
            kept_targets,
            kept_preds,
        )
    )

    print(
        f"[ID-only: reject "
        f"{prune_frac:.0%} most uncertain]"
    )

    print(
        f"  N ID: {n_id} | "
        f"N retained: {n_keep} | "
        f"N rejected: {n_prune}"
    )

    print(
        f"  Macro F1:          "
        f"{macro_f1_before:.4f} -> "
        f"{macro_f1_after:.4f}"
    )

    print(
        f"  Balanced Accuracy: "
        f"{balanced_acc_before:.4f} -> "
        f"{balanced_acc_after:.4f}"
    )

    return {
        "n_id": n_id,
        "n_retained": n_keep,
        "n_rejected": n_prune,
        "macro_f1_before": macro_f1_before,
        "macro_f1_after": macro_f1_after,
        "balanced_acc_before": balanced_acc_before,
        "balanced_acc_after": balanced_acc_after,
    }

def evaluate_full_pruning_composition(
    targets,
    uncertainty,
    prune_frac=0.10,
    unk_label=8,
):

    n_total = len(targets)
    n_prune = int(
        round(n_total * prune_frac)
    )

    # Reject the most uncertain samples
    rejected_idx = np.argsort(
        -uncertainty
    )[:n_prune]

    is_unk = targets == unk_label
    n_unk = int(is_unk.sum())

    n_rejected_unk = int(
        is_unk[rejected_idx].sum()
    )

    unk_caught = (
        n_rejected_unk / n_unk
        if n_unk > 0
        else np.nan
    )

    print(
        f"[Full test set: reject "
        f"{prune_frac:.0%} most uncertain]"
    )

    print(
        f"  N total: {n_total} | "
        f"N rejected: {n_prune}"
    )

    print(
        f"  UNK caught: "
        f"{n_rejected_unk}/{n_unk} "
        f"({unk_caught:.2%})"
    )

    return {
        "n_total": n_total,
        "n_rejected": n_prune,
        "unk_caught": unk_caught,
        "n_rejected_unk": n_rejected_unk,
    }

In [ ]:
preds_full, targets_full, uncertainty_full = (
    collect_test_predictions(
        model,
        test_loader,
        device,
    )
)

# Reject 10% most uncertain ID samples

id_results = evaluate_id_after_pruning(
    preds_full,
    targets_full,
    uncertainty_full,
    prune_frac=0.10,
    num_classes=8,
    unk_label=8,
)

# Reject 10% most uncertain samples from complete test set

full_results = evaluate_full_pruning_composition(
    targets_full,
    uncertainty_full,
    prune_frac=0.10,
    unk_label=8,
)